<a href="https://colab.research.google.com/github/2xsec/2xsec.github.io/blob/master/02_2_TFLite_microcontroller_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Setup

### 0-1. 필요한 파일 다운로드

먼저, TensorFlow Lite for Microcontroller repository를 clone합니다.

여기서는 실습을 위해 사전준비된 repository를 이용합니다: https://github.com/Jiw00n/TFLite_microcontroller

공식 TensorFlow Lite for Microcontroller GitHub repository 주소는 여기입니다: https://github.com/tensorflow/tflite-micro.git

In [ ]:
!git clone https://github.com/Jiw00n/TFLite_microcontroller

TFLite_microcontroller 폴더로 이동하여 code 컴파일에 필요한 third party 라이브러리들을 다운로드합니다.

In [ ]:
%cd TFLite_microcontroller/
!make -f tensorflow/lite/micro/tools/make/Makefile third_party_downloads

## 1. Person Detection (사람 인식) 모델

### 1-1. 예제 확인하기

이 실습에서는 grayscale 이미지로부터 사람을 인식 (person detect)하는 TF Lite 모델을 마이크로컨트롤러에 적용합니다.

person score에 사람이 있는 점수를, no person score에 사람이 없는 점수를 나타냅니다.

아래 스크립트를 실행시켜 예제를 살펴보세요.

In [ ]:
import matplotlib.pyplot as plt
from utils_tflite import run_inference

MODEL_PATH = 'tensorflow/lite/micro/models/person_detect.tflite'

img1_path = 'tensorflow/lite/micro/examples/person_detection/testdata/person.bmp'
img2_path = 'tensorflow/lite/micro/examples/person_detection/testdata/no_person.bmp'

img1, ps_1, nps_1 = run_inference(MODEL_PATH, img1_path)
img2, ps_2, nps_2 = run_inference(MODEL_PATH, img2_path)

plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.title(f'Person\n person score: {ps_1} \n no person score: {nps_1}', fontsize=18)
plt.imshow(img1, cmap='gray')
plt.subplot(1, 2, 2)
plt.title(f'No person\n person score: {ps_2} \n no person score: {nps_2}', fontsize=18)
plt.imshow(img2, cmap='gray')
plt.show()

## 2. TF Lite 모델을 C 배열로 변환하기

### 2-1. xxd 유틸리티를 활용한 C 배열 변환

Person detection 모델은 train이 완료되어 TF Lite 형식으로 저장 있습니다. 아래 경로에 모델이 위치합니다.

"`tensorflow/lite/micro/models/person_detect.tflite`"

이 TF Lite 모델을 xxd 유틸리티를 이용하여 C 배열로 변환 (파일이름: `person_detect_model_data.cc`)하세요.

여기서 변환된 C 모델 배열이 나중에 다른 코드들과 같이 컴파일되어 target device에 로드됩니다.

In [ ]:
# Install xxd if it is not available
!apt-get update && apt-get -qq install xxd

###### [빈칸] tensorflow/lite/micro/models/person_detect.tflite 모델을 ######
###### xxd를 이용하여 C 배열 (person_detect_model_data.cc)로 변환하세요. ######
!xxd -i tensorflow/lite/micro/models/person_detect.tflite > person_detect_model_data.cc

### 2-2. 변환된 C 배열 파일 확인

생성된 "`person_detect_model_data.cc`" 모델 C 배열 파일을 확인합니다.

In [ ]:
!head -n 20 person_detect_model_data.cc

In [ ]:
!tail -n 20 person_detect_model_data.cc

## 3. Inference 코드 작성하기

### 3-1. Colab 편집기에서 C 코드 작성

이제 마이크로컨트롤러에서 모델 inference를 수행하기 위한 code를 작성합니다.

Colab 왼쪽에 있는 메뉴에서 파일 tab를 클릭하고 "`TFLite_microcontroller/tensorflow/lite/micro/examples/person_detection/person_detection_test.cc`" 파일을 더블클릭하여 여세요. 화면 오른쪽에 코드 수정을 할 수 있는 편집창이 나옵니다.

<img src="https://drive.google.com/uc?export=view&id=1WEee_y_lTScrMxAZKXy-d3QRjJICkVQw" width="1000" hspace="0">

"`person_detection_test.cc`" 파일 안에 있는 한글 comment에 있는 내용에 따라 코드를 완성하세요. 수정된 내용은 자동저장됩니다.

코드를 전부 완성하고 다음 단계를 실행하세요.

## 4. 마이크로컨트롤러 바이너리 빌드

### 4-1. Linux용 컴파일

코드 작성이 완료되었으면, 작성한 "`person_detection_test.cc`" 파일을 Linux용 바이너리로 컴파일하고, 잘 동작하는지 실행해 봅니다.

In [ ]:
!make -f tensorflow/lite/micro/tools/make/Makefile test_person_detection_test_int8

### 4-2. ARM cortex-m7용 컴파일

Linux 머신에서 코드가 잘 동작하는 것을 확인하면, 이제 target microcontroller용 프로젝트를 생성하고, microcontroller 컴파일을 위한 toolcahin 등을 다운로드합니다.

이 실습에서는 ARM cortex-m7를 사용합니다.

In [ ]:
!make -f tensorflow/lite/micro/tools/make/Makefile \
TARGET=cortex_m_generic TARGET_ARCH=cortex-m7 \
generate_person_detection_int8_make_project

앞에서 C 배열로 변환한 모델을 컴파일 폴더로 복사합니다.

In [ ]:
!cp person_detect_model_data.cc tensorflow/lite/micro/tools/make/gen/linux_x86_64_default/genfiles/tensorflow/lite/micro/models/
!cp person_detect_model_data.cc tensorflow/lite/micro/tools/make/gen/cortex_m_generic_cortex-m7_default/genfiles/tensorflow/lite/micro/models/

모델이 폴더로 잘 이동하였는지 확인합니다.

In [ ]:
!ls -alh tensorflow/lite/micro/tools/make/gen/cortex_m_generic_cortex-m7_default/genfiles/tensorflow/lite/micro/models/person_detect_model_data.cc

이제 ARM cortex-m7r용 바이너리를 빌드합니다.

빌드 1

In [ ]:
!make -f tensorflow/lite/micro/tools/make/Makefile \
TARGET=cortex_m_generic \
TARGET_ARCH=cortex-m7 \
person_detection_int8

빌드 2 (세미호스팅; system call 오류 제거)

In [ ]:
!make -f tensorflow/lite/micro/tools/make/Makefile \
CXXFLAGS='--specs=rdimon.specs' \
TARGET=cortex_m_generic \
TARGET_ARCH=cortex-m7 \
person_detection_int8

아래 폴더에 Cortex-m7용 최종 바이너리가 "`person_detection_int8`" 파일로 생성된 것을 볼 수 있습니다.

In [ ]:
!ls -alh tensorflow/lite/micro/tools/make/gen/cortex_m_generic_cortex-m7_default/bin

In [ ]:
!make -f tensorflow/lite/micro/tools/make/Makefile clean